## In stacking what we  do is that we take 3-4 models then train them on the training data the output of that model is then use to train the other model that 

## But there is a problem in this method that when we training and testing on the same dataseet  which  is not good so there are some methods

# Blending

```mermaid
flowchart TD
    D["D"] -->|80| PTRAIN["D_train"]
    D -->|20| DTEST["D_test"]

    PTRAIN -->|80| TRAIN["D_train"]
    PTRAIN -->|20| VAL["D_valid"]

- ### after all the three models we make a prediction on D_valid  and then we will have a dataset in which we will have the prediction models and there actual values in the dataset
- ### After that we will train a meta model on the above data then we will use the complete artitecture for the prediction

# Stacking — Out-of-Fold (OOF) Predictions

**Stacking** is an ensemble learning technique where predictions from multiple **base models** are used as input features for a **meta-model**.

---

## 1. Train-Test Split

Suppose our original dataset contains **1000 samples**.

We first split the dataset into:

* **Training set:** 800 samples (80%)
* **Test set:** 200 samples (20%)

```mermaid
flowchart TD
    D["D<br/>1000 samples"]

    D -->|80%| TRAIN["D_train<br/>800 samples"]
    D -->|20%| TEST["D_test<br/>200 samples"]
```

> **Important:** `D_test` remains completely untouched until the final prediction stage.

---

## 2. Apply K-Fold Cross Validation

We apply **4-Fold Cross Validation** on `D_train`.

Since `D_train` contains 800 samples:

$$
\frac{800}{4} = 200
$$

Therefore, each fold contains **200 samples**.

```text
D_train = 800 samples

┌────────┬────────┬────────┬────────┐
│ Fold 1 │ Fold 2 │ Fold 3 │ Fold 4 │
│  200   │  200   │  200   │  200   │
└────────┴────────┴────────┴────────┘
```

During each iteration, one fold is used for validation and the remaining three folds are used for training.

### Fold 1

```text
┌────────┬────────┬────────┬────────┐
│ VALID  │ TRAIN  │ TRAIN  │ TRAIN  │
│  200   │  200   │  200   │  200   │
└────────┴────────┴────────┴────────┘
```

### Fold 2

```text
┌────────┬────────┬────────┬────────┐
│ TRAIN  │ VALID  │ TRAIN  │ TRAIN  │
│  200   │  200   │  200   │  200   │
└────────┴────────┴────────┴────────┘
```

### Fold 3

```text
┌────────┬────────┬────────┬────────┐
│ TRAIN  │ TRAIN  │ VALID  │ TRAIN  │
│  200   │  200   │  200   │  200   │
└────────┴────────┴────────┴────────┘
```

### Fold 4

```text
┌────────┬────────┬────────┬────────┐
│ TRAIN  │ TRAIN  │ TRAIN  │ VALID  │
│  200   │  200   │  200   │  200   │
└────────┴────────┴────────┴────────┘
```

---

## 3. Train the Base Models

We use three base models:

* **Linear Regression (LR)**
* **Decision Tree (DT)**
* **K-Nearest Neighbors (KNN)**

For every fold, each model is trained on the training portion and makes predictions on the validation portion.

```mermaid
flowchart LR
    DATA["Training Fold"]

    DATA --> LR["Linear Regression"]
    DATA --> DT["Decision Tree"]
    DATA --> KNN["KNN"]

    LR --> LRP["LR Prediction"]
    DT --> DTP["DT Prediction"]
    KNN --> KNNP["KNN Prediction"]
```

With 4 folds and 3 base models:

$$
4 \times 3 = 12
$$

So, we perform **12 model fits** during cross-validation.

---

## 4. Generate Out-of-Fold Predictions

For every fold, the models predict only on the validation data.

Therefore, each training sample receives a prediction from a model that **has not seen that sample during training**.

For example:

```text
Fold 1 → Predictions for 200 samples
Fold 2 → Predictions for 200 samples
Fold 3 → Predictions for 200 samples
Fold 4 → Predictions for 200 samples
```

After completing all four folds, we have predictions for all:

$$
200 + 200 + 200 + 200 = 800
$$

training samples.

These are called **Out-of-Fold (OOF) Predictions**.

---

## 5. Create the OOF Dataset

The predictions from the base models are combined to create a new dataset.

| LR Prediction | DT Prediction | KNN Prediction | Actual Target |
| :-----------: | :-----------: | :------------: | :-----------: |
|    $p_{LR}$   |    $p_{DT}$   |    $p_{KNN}$   |      $y$      |
|    $p_{LR}$   |    $p_{DT}$   |    $p_{KNN}$   |      $y$      |
|    $p_{LR}$   |    $p_{DT}$   |    $p_{KNN}$   |      $y$      |
|    $\vdots$   |    $\vdots$   |    $\vdots$    |    $\vdots$   |

Therefore:

```text
OOF Dataset
     │
     ├── LR Prediction
     ├── DT Prediction
     ├── KNN Prediction
     └── Actual Target
```

The OOF dataset has:

$$
800 \times 4
$$

dimensions.

The first three columns are used as features for the meta-model.

---

## 6. Why Do We Need OOF Predictions?

We **cannot simply train the base models on the training data and then predict on the same training data**.

That would result in predictions generated from data that the models have already seen.

### ❌ Incorrect Approach

```mermaid
flowchart LR
    DATA["Training Data"] --> MODEL["Train Base Model"]
    MODEL --> PRED["Predict on Same Data"]
    PRED --> META["Meta Model"]
```

This can lead to **data leakage** and overly optimistic predictions.

### ✅ Correct Approach

```mermaid
flowchart LR
    TRAIN["Training Fold"] --> MODEL["Train Base Model"]
    VALID["Validation Fold"] --> MODEL
    MODEL --> OOF["OOF Prediction"]
    OOF --> META["Meta Model"]
```

The validation samples are **unseen by the base model**, so their predictions provide a more realistic representation of how the model behaves on unseen data.

---

## 7. Train the Meta-Model

Once all OOF predictions have been generated, they are used as input features for a second-level model called the **meta-model**.

For example, we can use a **Random Forest Regressor (RFR)**.

```mermaid
flowchart LR
    LR["LR Prediction"]
    DT["DT Prediction"]
    KNN["KNN Prediction"]

    LR --> META["Random Forest<br/>Meta-Model"]
    DT --> META
    KNN --> META

    META --> FINAL["Final Prediction"]
```

The meta-model learns how to combine the predictions of the different base models.

---

## 8. Train Base Models on the Complete Training Set

After generating the OOF predictions, we train each base model again using **all 800 training samples**.

```mermaid
flowchart LR
    TRAIN["D_train<br/>800 samples"]

    TRAIN --> LR["Linear Regression"]
    TRAIN --> DT["Decision Tree"]
    TRAIN --> KNN["KNN"]
```

These fully trained models are then used to make predictions on the previously untouched `D_test`.

```mermaid
flowchart LR
    TEST["D_test<br/>200 samples"]

    TEST --> LR["Trained LR"]
    TEST --> DT["Trained DT"]
    TEST --> KNN["Trained KNN"]

    LR --> P1["LR Test Prediction"]
    DT --> P2["DT Test Prediction"]
    KNN --> P3["KNN Test Prediction"]
```

The three predictions are then passed to the trained meta-model.

---

# Complete Stacking Pipeline

```mermaid
flowchart TD
    D["D<br/>1000 Samples"]

    D --> TRAIN["D_train<br/>800 Samples"]
    D --> TEST["D_test<br/>200 Samples"]

    TRAIN --> CV["4-Fold Cross Validation"]

    CV --> LR["Linear Regression"]
    CV --> DT["Decision Tree"]
    CV --> KNN["KNN"]

    LR --> OOF1["LR OOF Predictions"]
    DT --> OOF2["DT OOF Predictions"]
    KNN --> OOF3["KNN OOF Predictions"]

    OOF1 --> META["Meta Model<br/>Random Forest"]
    OOF2 --> META
    OOF3 --> META

    TRAIN --> LR2["Train LR on Full D_train"]
    TRAIN --> DT2["Train DT on Full D_train"]
    TRAIN --> KNN2["Train KNN on Full D_train"]

    TEST --> LR2
    TEST --> DT2
    TEST --> KNN2

    LR2 --> TP1["LR Test Prediction"]
    DT2 --> TP2["DT Test Prediction"]
    KNN2 --> TP3["KNN Test Prediction"]

    TP1 --> META2["Trained Meta Model"]
    TP2 --> META2
    TP3 --> META2

    META2 --> FINAL["Final Prediction"]
```

---

## Key Numbers

| Component                 |                   Value |
| ------------------------- | ----------------------: |
| Original Dataset          |            1000 samples |
| Training Set              |             800 samples |
| Test Set                  |             200 samples |
| Number of Folds           |                       4 |
| Samples per Fold          |                     200 |
| Training Samples per Fold |                     600 |
| Base Models               |                       3 |
| CV Model Fits             |                      12 |
| OOF Samples               |                     800 |
| OOF Features              |                       3 |
| Meta-Model                | Random Forest Regressor |

---

## Final Concept

The overall idea of stacking is:

$$
\boxed{
\text{Base Models}
\rightarrow
\text{OOF Predictions}
\rightarrow
\text{Meta-Model}
\rightarrow
\text{Final Prediction}
}
$$

The important point is that **OOF predictions prevent the meta-model from learning from predictions generated on data that the base models have already seen**.

This allows the meta-model to learn how to optimally combine the different base models.


# Multi-Layer Stacking

**Multi-Layer Stacking** extends traditional stacking by using **multiple levels of base models**.

Instead of directly sending the predictions of the first-layer models to a final meta-model, the predictions are passed through another layer of models. This process can be repeated for multiple layers.

---

## 1. Basic Architecture

Suppose we have three models in the first layer:

* $m_1$
* $m_2$
* $m_3$

Their predictions are passed to a second layer containing:

* $m_4$
* $m_5$
* $m_6$

Finally, the predictions from the second layer are passed to a final model $m_7$.

```mermaid
flowchart LR
    M1["m₁"]
    M2["m₂"]
    M3["m₃"]

    M4["m₄"]
    M5["m₅"]
    M6["m₆"]

    M7["m₇<br/>Meta Model"]

    M1 --> M4
    M1 --> M5
    M1 --> M6

    M2 --> M4
    M2 --> M5
    M2 --> M6

    M3 --> M4
    M3 --> M5
    M3 --> M6

    M4 --> M7
    M5 --> M7
    M6 --> M7
```

The connections between the layers are **fully connected**, meaning every model in one layer can provide its prediction to every model in the next layer.

---

# 2. Dataset Split

Assume the original dataset contains **1000 samples**.

We first split it into:

* `D_train` → 900 samples
* `D_test` → 100 samples

```mermaid
flowchart TD
    D["D<br/>1000 Samples"]

    D -->|90%| TRAIN["D_train<br/>900 Samples"]
    D -->|10%| TEST["D_test<br/>100 Samples"]
```

The test set is kept completely separate from the stacking process.

> **Important:** `D_test` should not be used to train any model.

---

# 3. Divide `D_train` into Folds

The training dataset is divided into **3 folds**.

Since there are 900 training samples:

$$
\frac{900}{3}=300
$$

Therefore:

```text
D_train = 900 samples

┌────────────┬────────────┬────────────┐
│   D_T1     │   D_T2     │   D_T3     │
│   300      │   300      │   300      │
└────────────┴────────────┴────────────┘
```

For each iteration, one fold is used for validation while the remaining folds are used for training.

---

## Fold 1

```text
┌────────────┬────────────┬────────────┐
│  VALID     │   TRAIN    │   TRAIN    │
│   300      │    300     │    300     │
└────────────┴────────────┴────────────┘
```

## Fold 2

```text
┌────────────┬────────────┬────────────┐
│   TRAIN    │   VALID    │   TRAIN    │
│    300     │    300     │    300     │
└────────────┴────────────┴────────────┘
```

## Fold 3

```text
┌────────────┬────────────┬────────────┐
│   TRAIN    │   TRAIN    │   VALID    │
│    300     │    300     │    300     │
└────────────┴────────────┴────────────┘
```

---

# 4. Layer 1 — Base Models

The first layer contains three models:

$$
m_1,\quad m_2,\quad m_3
$$

Each model is trained using the training portion of each fold and generates predictions for the corresponding validation fold.

```mermaid
flowchart LR
    DATA["Training Data"]

    DATA --> M1["m₁"]
    DATA --> M2["m₂"]
    DATA --> M3["m₃"]

    M1 --> P1["m₁ predictions"]
    M2 --> P2["m₂ predictions"]
    M3 --> P3["m₃ predictions"]
```

For every validation sample, we therefore obtain:

$$
[m_1(x),m_2(x),m_3(x)]
$$

These predictions become the **features for the next layer**.

---

# 5. Creating the First-Layer Prediction Dataset

Suppose the first-layer models generate predictions for 300 validation samples.

The resulting dataset looks like:

| $m_1$ Prediction | $m_2$ Prediction | $m_3$ Prediction |  Target  |
| :--------------: | :--------------: | :--------------: | :------: |
|       $p_1$      |       $p_2$      |       $p_3$      |    $y$   |
|       $p_1$      |       $p_2$      |       $p_3$      |    $y$   |
|     $\vdots$     |     $\vdots$     |     $\vdots$     | $\vdots$ |

After completing all folds, we obtain predictions for all **900 training samples**.

Therefore, the new feature matrix is approximately:

$$
X_{\text{new}} \in \mathbb{R}^{900\times3}
$$

where the three features are:

```text
m₁_prediction
m₂_prediction
m₃_prediction
```

These are **Out-of-Fold (OOF) predictions**.

---

# 6. Layer 2

The OOF predictions generated by the first layer are passed to the second layer.

The second layer contains:

$$
m_4,\quad m_5,\quad m_6
$$

Each of these models receives the predictions from **all three models of Layer 1**.

```mermaid
flowchart LR
    M1["m₁"]
    M2["m₂"]
    M3["m₃"]

    M4["m₄"]
    M5["m₅"]
    M6["m₆"]

    M1 --> M4
    M1 --> M5
    M1 --> M6

    M2 --> M4
    M2 --> M5
    M2 --> M6

    M3 --> M4
    M3 --> M5
    M3 --> M6
```

Therefore, each second-layer model receives:

$$
X_2 =
[
m_1(x),
m_2(x),
m_3(x)
]
$$

and learns to produce a new prediction.

---

# 7. Generate Second-Layer Predictions

The second layer generates:

$$
m_4(x),\quad m_5(x),\quad m_6(x)
$$

These predictions form another feature matrix:

| $m_4$ Prediction | $m_5$ Prediction | $m_6$ Prediction |
| :--------------: | :--------------: | :--------------: |
|       $p_4$      |       $p_5$      |       $p_6$      |
|       $p_4$      |       $p_5$      |       $p_6$      |
|     $\vdots$     |     $\vdots$     |     $\vdots$     |

Now the second-layer output becomes the input to the final model.

---

# 8. Final Layer — Meta Model

The final model is:

$$
m_7
$$

It receives the predictions generated by the second layer:

$$
[
m_4(x),
m_5(x),
m_6(x)
]
$$

and produces the final prediction.

```mermaid
flowchart LR
    M4["m₄"]
    M5["m₅"]
    M6["m₆"]

    M7["m₇<br/>Final Meta Model"]

    M4 --> M7
    M5 --> M7
    M6 --> M7

    M7 --> OUT["Final Prediction"]
```

---

# 9. Complete Multi-Layer Stacking

The entire architecture can be represented as:

```mermaid
flowchart LR
    subgraph L1["Layer 1"]
        M1["m₁"]
        M2["m₂"]
        M3["m₃"]
    end

    subgraph L2["Layer 2"]
        M4["m₄"]
        M5["m₅"]
        M6["m₆"]
    end

    subgraph L3["Layer 3"]
        M7["m₇"]
    end

    M1 --> M4
    M1 --> M5
    M1 --> M6

    M2 --> M4
    M2 --> M5
    M2 --> M6

    M3 --> M4
    M3 --> M5
    M3 --> M6

    M4 --> M7
    M5 --> M7
    M6 --> M7

    M7 --> P["Final Prediction"]
```

Conceptually:

$$
\boxed{
\text{Original Features}
\rightarrow
\text{Layer 1}
\rightarrow
\text{Layer 2}
\rightarrow
\text{Meta Model}
\rightarrow
\text{Final Prediction}
}
$$

---

# 10. Why Use Multiple Layers?

In traditional stacking:

$$
\text{Base Models}
\rightarrow
\text{Meta Model}
$$

In multi-layer stacking:

$$
\text{Layer 1}
\rightarrow
\text{Layer 2}
\rightarrow
\text{Layer 3}
\rightarrow
\cdots
\rightarrow
\text{Final Model}
$$

Each layer learns a new representation of the predictions produced by the previous layer.

For example:

```text
Original Features
       ↓
 ┌───────────────┐
 │ m₁  m₂  m₃    │  ← Layer 1
 └───────────────┘
       ↓
 ┌───────────────┐
 │ m₄  m₅  m₆    │  ← Layer 2
 └───────────────┘
       ↓
 ┌───────────────┐
 │      m₇       │  ← Final Layer
 └───────────────┘
       ↓
Final Prediction
```

---

# 11. Avoiding Data Leakage

The most important part of multi-layer stacking is **preventing data leakage**.

A model should not generate predictions for samples that it was trained on and then pass those predictions to the next layer.

Instead:

```mermaid
flowchart LR
    TRAIN["Training Fold"] --> MODEL["Train Model"]
    VALID["Validation Fold"] --> MODEL
    MODEL --> OOF["OOF Prediction"]
    OOF --> NEXT["Next Layer"]
```

The prediction for a sample must be generated by a model that **did not see that sample during training**.

This is why **Out-of-Fold predictions** are used between layers.

---

# 12. Prediction Flow for the Test Set

After the stacking layers have been trained, the test set is passed through the entire hierarchy.

```mermaid
flowchart LR
    TEST["D_test"]

    TEST --> M1["Trained m₁"]
    TEST --> M2["Trained m₂"]
    TEST --> M3["Trained m₃"]

    M1 --> M4["m₄"]
    M2 --> M4
    M3 --> M4

    M1 --> M5["m₅"]
    M2 --> M5
    M3 --> M5

    M1 --> M6["m₆"]
    M2 --> M6
    M3 --> M6

    M4 --> M7["m₇"]
    M5 --> M7
    M6 --> M7

    M7 --> FINAL["Final Test Prediction"]
```

The important point is that the **same sequence of transformations used during training must be followed during testing**.

---

# 13. Summary

|  Layer  |     Models    | Input               |
| :-----: | :-----------: | :------------------ |
| Layer 1 | $m_1,m_2,m_3$ | Original features   |
| Layer 2 | $m_4,m_5,m_6$ | Layer 1 predictions |
| Layer 3 |     $m_7$     | Layer 2 predictions |

The core idea is:

$$
\boxed{
X
\rightarrow
[m_1,m_2,m_3]
\rightarrow
[m_4,m_5,m_6]
\rightarrow
m_7
\rightarrow
\hat{y}
}
$$

### Key Takeaway

> **Multi-Layer Stacking recursively learns from model predictions. Each layer takes the OOF predictions from the previous layer as its input, allowing the ensemble to learn increasingly complex combinations of the predictions.**
